# Three-axis uncertainty for audio analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sensein/senselab/blob/main/tutorials/audio/uncertainty_axes.ipynb)

This notebook walks through `senselab.audio.workflows.audio_analysis` — the three-axis uncertainty workflow used by `scripts/analyze_audio.py`. The workflow takes the per-task pipeline output (diarization, ASR, scene classification, alignment, PPG) and produces three per-bucket time series:

- **presence_uncertainty** — was there a speaker?
- **identity_uncertainty** — was it the same speaker?
- **utterance_uncertainty** — what was said?

Each axis collapses every contributing model's vote into one `[0, 1]` scalar per bucket. The result is 9 parquets (3 axes × 2 passes + 3 raw-vs-enhanced deltas) when the script runs end-to-end, or just the in-memory `AxisResult` objects when you call the workflow directly.

See `specs/20260508-173136-compare-uncertainty/` for the full design.

In [ ]:
# Install senselab
!pip install -q uv
!uv pip install --pre --system 'senselab[nlp]'

> **⚠️ Restart runtime after install**
>
> The install may upgrade packages already imported by Colab. After it finishes, choose **Runtime → Restart session** and continue from the next cell.

In [ ]:
from pathlib import Path
from types import SimpleNamespace

from senselab.audio.data_structures import Audio
from senselab.audio.workflows.audio_analysis import (
    BucketGrid,
    build_aligned_timeline_plot,
    build_disagreements_index,
    compute_uncertainty_axes,
)

## 1. Build a synthetic `passes` summary

The workflow reads the dict-of-dicts shape that `scripts/analyze_audio.py`'s `run_pass` produces — keyed by task, then by `"by_model"` for multi-model tasks. For this tutorial we hand-build a tiny version so we don't have to load any audio models. In a real run, you'd pass the cached output from analyze_audio's per-task pipeline directly.

The synthetic clip below is 4 s long with two diar models agreeing on speech in `[0, 1]` and `[1, 4]`, and two ASR models with one transcript edit in the second half (`granite` says `"world!!"` while `whisper` says `"world"`).

In [ ]:
def diar_block(segments):
    segs = [SimpleNamespace(start=s, end=e, speaker=spk, text="") for s, e, spk in segments]
    return {"status": "ok", "result": [segs], "cache_key": "diar_k"}


def asr_block(chunks, avg_logprob=-0.2):
    chunk_objs = [
        SimpleNamespace(start=s, end=e, text=t, avg_logprob=avg_logprob, no_speech_prob=0.05) for s, e, t in chunks
    ]
    line = SimpleNamespace(
        text=" ".join(t for _, _, t in chunks),
        chunks=chunk_objs,
        start=chunks[0][0],
        end=chunks[-1][1],
        avg_logprob=avg_logprob,
    )
    return {"status": "ok", "result": [line], "cache_key": "asr_k"}


diar_segs = [(0.0, 1.0, "SPEAKER_00"), (1.0, 4.0, "SPEAKER_01")]
raw_pass = {
    "duration_s": 4.0,
    "diarization": {
        "by_model": {
            "pyannote": diar_block(diar_segs),
            "sortformer": diar_block(diar_segs),
        }
    },
    "asr": {
        "by_model": {
            "whisper": asr_block([(0.0, 1.0, "hello"), (1.0, 4.0, "world")]),
            "granite": asr_block([(0.0, 1.0, "hello"), (1.0, 4.0, "world!!")]),
        }
    },
}
enh_pass = {
    "duration_s": 4.0,
    "diarization": {
        "by_model": {
            "pyannote": diar_block(diar_segs),
            "sortformer": diar_block(diar_segs),
        }
    },
    "asr": {
        "by_model": {
            "whisper": asr_block([(0.0, 1.0, "hello"), (1.0, 4.0, "world")]),
            "granite": asr_block([(0.0, 1.0, "hello"), (1.0, 4.0, "world")]),
        }
    },
}

## 2. Run the workflow

`compute_uncertainty_axes` is a pure function: in-memory inputs → in-memory `AxisResult` objects. The script wraps it with cache lookup, parquet writing, LS bundle assembly, disagreements.json, and the timeline plot.

In [ ]:
import torch

audio = {
    pl: Audio(waveform=torch.zeros(1, int(4.0 * 16000)), sampling_rate=16000) for pl in ("raw_16k", "enhanced_16k")
}

axis_results, incomparable = compute_uncertainty_axes(
    passes={"raw_16k": raw_pass, "enhanced_16k": enh_pass},
    grid=BucketGrid(win_length=0.5, hop_length=0.5),
    params={"win_length": 0.5, "hop_length": 0.5, "aggregator": "min"},
    audio=audio,
    speaker_embedding_models=[],  # Skip embeddings in the tutorial — keeps it model-free.
    aggregator="min",
    speech_presence_labels=["Speech"],
)

print("axis_results keys:", sorted(axis_results.keys()))
print("incomparable_reasons:", incomparable)

## 3. Read the per-bucket uncertainties

Each `AxisResult` carries a list of `UncertaintyRow` objects. Per-row fields are documented in `contracts/uncertainty-row.parquet.md`.

In [ ]:
for axis in ("presence", "identity", "utterance"):
    print(f"\n── raw_16k / {axis} ──")
    for r in axis_results[("raw_16k", axis)].rows[:6]:
        u = r.aggregated_uncertainty
        print(
            f"  [{r.start:5.2f}, {r.end:5.2f})  u={u if u is None else round(u, 3):>5}  models={r.contributing_models}"
        )

## 4. Build the disagreements index

`build_disagreements_index` ranks rows across all 9 axis_results by `aggregated_uncertainty desc`, with axis-priority tiebreak (utterance > identity > presence). The output is the JSON shape documented in `contracts/disagreements.json.md`.

In [ ]:
idx = build_disagreements_index(
    axis_results=axis_results,
    top_n=10,
    run_dir=Path("."),
    config={
        "top_n": 10,
        "aggregator": "min",
        "phoneme_disagreement_threshold": 0.5,
        "bucket_grid": {"win_length": 0.5, "hop_length": 0.5},
    },
    incomparable_reasons=incomparable,
    models_without_native_signal=["pyannote", "sortformer"],
)

print(f"top_n={len(idx['entries'])} | totals: {idx['totals']}")
for entry in idx["entries"][:5]:
    print(f"  #{entry['rank']}  {entry['axis']:9s}  {entry['pass']:18s}  u={entry['aggregated_uncertainty']}")

## 5. Render the timeline plot

`build_aligned_timeline_plot` produces a 5-row figure: presence / identity / utterance overlaid raw + enhanced, plus a raw-vs-enhanced delta strip and a reference row (raw diar speakers + raw ASR token spans).

In [ ]:
tmp_dir = Path(".") / "_tutorial_out"
tmp_dir.mkdir(exist_ok=True)

plot_path = build_aligned_timeline_plot(
    run_dir=tmp_dir,
    axis_results=axis_results,
    duration_s=4.0,
    grid_hop=0.5,
    title="tutorial — synthetic 4 s clip",
)
print(f"plot: {plot_path} ({plot_path.stat().st_size} bytes)" if plot_path else "no plot")

from IPython.display import Image

Image(filename=str(plot_path)) if plot_path else None

## 6. Run on real audio

To reproduce this on a real conversation rather than synthetic data, run the script end-to-end on `tutorial_audio_files/english_conversation_higgs_audio_v2.wav` (a 21.5 s multi-speaker clip checked into the repo):

```bash
uv run python scripts/analyze_audio.py tutorial_audio_files/english_conversation_higgs_audio_v2.wav
```

The script writes the 9 parquets, `disagreements.json`, `timeline.png`, and the LS bundle into `artifacts/analyze_audio/<run_id>/`. The same `compute_uncertainty_axes` you called above is what the script wraps.